# Option B: Retraining Stability (AE vs UMAP)

**Experiment Design:**
1. Train AE 10 times with different random seeds
2. Train UMAP 10 times with different random seeds
3. For each method, compute pairwise correlations between all embedding pairs (after Procrustes alignment)

**Metrics:**
- Mean pairwise correlation across runs
- Per-OA standard deviation across runs
- Identifies which areas have high embedding variance

**Expected Result:**
- **AE**: Correlations > 0.95 (highly reproducible)
- **UMAP**: Lower and more variable, especially in sparse regions

**Real-world relevance:** Answers "If two researchers train the model, do they get the same answer?"

## 1. Setup and Imports

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import yaml
from torchgeodemo import autoencoder_train_latent
from scipy.spatial import procrustes
from scipy.stats import pearsonr
import umap
import geopandas as gpd
import os
import pickle
from tqdm import tqdm
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.7.1+cu126
CUDA available: True


## 2. Configuration

In [8]:
# Helper function for layer size generation (from notebook 2)
def gen_layer_sizes(input_size, latent_size, num_layers, scaling_type="lin"):
    """Generate encoder/decoder layer widths."""
    if scaling_type == "mul":
        scale = (latent_size / input_size) ** (1 / (num_layers - 1))
        return [int(input_size * scale ** i) for i in range(1, num_layers)]
    if scaling_type == "lin":
        step = (latent_size - input_size) / (num_layers - 1)
        return [int(input_size + step * i) for i in range(1, num_layers)]
    raise ValueError("Invalid scaling type. Use 'mul' or 'lin'.")

# Paths
data_path = "../data/census_data/engcensus_cleaned_scaled.parquet"
geofile_path = "../data/geofiles/Output_Areas_(December_2021)_Boundaries_EW_BFE_(V9)_and_RUC.geojson"
output_dir = "./plots/retraining_stability/"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/data/", exist_ok=True)
os.makedirs(f"{output_dir}/models/", exist_ok=True)
os.makedirs(f"{output_dir}/models/yamls/", exist_ok=True)
os.makedirs(f"{output_dir}/yamls/", exist_ok=True)

# Experiment parameters
latent_dim = 100  # Use 100D linear AE
n_runs = 10  # Number of retraining runs
base_seed = 20210321  # Base random seed

# Training parameters (match notebook 2)
n_epochs = 250
batch_size = 0.01  # 1% of data
scaling_type = "lin"  # Linear layer scaling

# UMAP parameters
umap_n_neighbors = 15
umap_min_dist = 0.1
umap_metric = 'euclidean'

print("Configuration:")
print(f"  Latent dimension: {latent_dim}")
print(f"  Number of runs: {n_runs}")
print(f"  Base seed: {base_seed}")
print(f"  Training epochs: {n_epochs}")
print(f"  Batch size: {batch_size} ({int(batch_size*100)}% of data)")
print(f"  Layer scaling: {scaling_type}")
print(f"  Output directory: {output_dir}")

Configuration:
  Latent dimension: 100
  Number of runs: 10
  Base seed: 20210321
  Training epochs: 250
  Batch size: 0.01 (1% of data)
  Layer scaling: lin
  Output directory: ./plots/retraining_stability/


## 3. Utility Functions

In [9]:
def procrustes_align(X_source, X_target):
    """
    Align X_source to X_target using Procrustes transformation.
    Returns: aligned X_source, disparity
    """
    mtx1, mtx2, disparity = procrustes(X_target, X_source)
    return mtx1, disparity

def compute_pairwise_correlations(embeddings_list):
    """
    Compute pairwise correlations between all embedding pairs.
    
    Parameters:
    - embeddings_list: list of embedding arrays (each shape: n_samples x n_dims)
    
    Returns:
    - mean_correlation: mean across all pairs
    - correlation_matrix: n_runs x n_runs matrix of correlations
    - all_correlations: flat array of all pairwise correlations
    """
    n_runs = len(embeddings_list)
    correlation_matrix = np.ones((n_runs, n_runs))
    all_correlations = []
    
    for i, j in combinations(range(n_runs), 2):
        # Align j to i using Procrustes
        aligned_j, _ = procrustes_align(embeddings_list[j], embeddings_list[i])
        
        # Compute per-dimension correlations
        dim_corrs = []
        for dim in range(embeddings_list[i].shape[1]):
            corr, _ = pearsonr(embeddings_list[i][:, dim], aligned_j[:, dim])
            dim_corrs.append(corr)
        
        mean_corr = np.mean(dim_corrs)
        correlation_matrix[i, j] = mean_corr
        correlation_matrix[j, i] = mean_corr
        all_correlations.append(mean_corr)
    
    return np.mean(all_correlations), correlation_matrix, np.array(all_correlations)

def compute_per_oa_std(embeddings_list):
    """
    Compute per-OA standard deviation across runs.
    First aligns all embeddings to the first run using Procrustes.
    
    Returns: array of shape (n_samples,) with std for each OA
    """
    n_runs = len(embeddings_list)
    reference = embeddings_list[0]
    
    # Align all to reference
    aligned_embeddings = [reference]
    for i in range(1, n_runs):
        aligned, _ = procrustes_align(embeddings_list[i], reference)
        aligned_embeddings.append(aligned)
    
    # Stack and compute std across runs
    stacked = np.stack(aligned_embeddings, axis=0)  # shape: (n_runs, n_samples, n_dims)
    per_oa_std = np.mean(stacked.std(axis=0), axis=1)  # mean std across dimensions
    
    return per_oa_std

print("Utility functions defined")

Utility functions defined


## 4. Helper Functions for AE Training via YAML

In [ ]:
def train_ae_via_yaml(data_df, run_name, latent_dim, working_dir, seed,
                      n_epochs=250, batch_size=0.01, scaling_type="lin"):
    """
    Train autoencoder using torchgeodemo with YAML configuration.
    
    Parameters:
    - data_df: DataFrame with data (must have 'OA' column)
    - run_name: name for this run (e.g., 'run_0', 'run_1')
    - latent_dim: bottleneck dimension
    - working_dir: directory for outputs
    - seed: random seed for this run
    - n_epochs: training epochs
    - batch_size: batch size fraction
    - scaling_type: 'lin' or 'mul' for layer scaling
    
    Returns: embeddings (numpy array)
    """
    print(f"[train_ae_via_yaml] {run_name}: start (seed={seed})")
    torch.manual_seed(seed)
    
    if 'OA' not in data_df.columns:
        print(f"[train_ae_via_yaml] {run_name}: adding OA index column")
        data_df = data_df.reset_index()
    
    input_dim = data_df.shape[1] - 1
    print(f"[train_ae_via_yaml] {run_name}: input_dim={input_dim}, latent_dim={latent_dim}")
    encoder_sizes = gen_layer_sizes(input_dim, latent_dim, num_layers=4, scaling_type=scaling_type)
    print(f"[train_ae_via_yaml] {run_name}: encoder sizes {encoder_sizes}")
    
    yaml_config = {
        "data": {
            "source": "TEMP",
            "nickname": f"stability_{run_name}",
            "id_col": "OA"
        },
        "working_dir": working_dir,
        "autoencoder": {
            "nickname": f"ae_{latent_dim}d_{run_name}",
            "version": "1",
            "save_latent": "csv",
            "max_epochs": n_epochs,
            "batch_size": batch_size,
            "use_covariance_loss": False,
            "random_seed": seed,
            "encoder": {
                "sizes": encoder_sizes,
                "activation": "LeakyReLU"
            },
            "decoder": {
                "sizes": encoder_sizes[::-1],
                "activation": "LeakyReLU"
            }
        }
    }
    
    temp_data_path = f"{working_dir}/temp_data_{run_name}.parquet"
    print(f"[train_ae_via_yaml] {run_name}: writing data to {temp_data_path}")
    data_df.to_parquet(temp_data_path)
    yaml_config["data"]["source"] = temp_data_path
    
    yaml_dir = f"{working_dir}/yamls"
    os.makedirs(yaml_dir, exist_ok=True)
    config_path = f"{yaml_dir}/config_{run_name}_{latent_dim}d.yaml"
    print(f"[train_ae_via_yaml] {run_name}: writing YAML to {config_path}")
    with open(config_path, 'w') as f:
        yaml.dump(yaml_config, f, default_flow_style=False)
    
    print(f"[train_ae_via_yaml] {run_name}: launching training")
    autoencoder_train_latent.main(config_path, create_latent=True, save_reco_error=False, verbose=False)
    print(f"[train_ae_via_yaml] {run_name}: training complete")
    
    model_nickname = f"stability_{run_name}_ae_{latent_dim}d_{run_name}_v1"
    latent_csv_path = f"{working_dir}/{model_nickname}/{model_nickname}__latent.csv"
    
    if os.path.exists(latent_csv_path):
        print(f"[train_ae_via_yaml] {run_name}: loading latent CSV {latent_csv_path}")
        latent_df = pd.read_csv(latent_csv_path, index_col="OA")
        embeddings = latent_df.values
    else:
        model_path = f"{working_dir}/{model_nickname}/{model_nickname}__model.pth"
        print(f"[train_ae_via_yaml] {run_name}: latent CSV missing, encoding manually via {model_path}")
        embeddings = load_ae_and_encode(model_path, data_df.drop(columns=['OA']))
    
    print(f"[train_ae_via_yaml] {run_name}: embeddings shape {embeddings.shape}")
    return embeddings

def load_ae_and_encode(model_path, X_data):
    """Load trained AE model and encode data"""
    from torchgeodemo.models import AutoEncoder
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = torch.load(model_path, map_location=device, weights_only=False)
    model.eval()
    
    if isinstance(X_data, pd.DataFrame):
        X_np = X_data.values
    else:
        X_np = X_data
    
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_np).to(device)
        embeddings = model.encode(X_tensor).cpu().numpy()
    
    return embeddings

print("YAML-based training and loading functions defined")

YAML-based training and loading functions defined


## 5. Load Data

In [11]:
# Load census data as DataFrame
df = pd.read_parquet(data_path)
print(f"Census data shape: {df.shape}")

# Ensure OA column exists
if 'OA' not in df.columns:
    df = df.reset_index()

# Also create numpy version for UMAP
X_full = df.drop(columns=['OA']).values
oa_ids = df['OA'].values

print(f"\nData summary:")
print(f"  Total OAs: {len(df)}")
print(f"  Variables: {X_full.shape[1]}")
print(f"  Data range: [{X_full.min():.4f}, {X_full.max():.4f}]")

Census data shape: (188880, 409)

Data summary:
  Total OAs: 188880
  Variables: 408
  Data range: [0.0000, 1.0000]


## 6. Experiment: Train AE Multiple Times

In [12]:
print("=" * 80)
print(f"STEP 1: Train AE {n_runs} times with different random seeds using torchgeodemo")
print("=" * 80)

ae_embeddings_list = []
ae_seeds = [base_seed + i * 1000 for i in range(n_runs)]

for run_idx, seed in enumerate(tqdm(ae_seeds, desc="Training AE runs")):
    run_name = f"run_{run_idx}"
    print(f"\nRun {run_idx + 1}/{n_runs} (seed={seed})")
    
    embeddings = train_ae_via_yaml(
        data_df=df.copy(),
        run_name=run_name,
        latent_dim=latent_dim,
        working_dir=f"{output_dir}/models",
        seed=seed,
        n_epochs=n_epochs,
        batch_size=batch_size,
        scaling_type=scaling_type
    )
    
    ae_embeddings_list.append(embeddings)
    print(f"  Embeddings shape: {embeddings.shape}")

print(f"\n✓ Trained {n_runs} AE models")
print(f"  Each embedding: {ae_embeddings_list[0].shape}")

STEP 1: Train AE 10 times with different random seeds using torchgeodemo


Training AE runs:   0%|          | 0/10 [00:00<?, ?it/s]


Run 1/10 (seed=20210321)


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A500 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type    | Params | Mode 
--------------------------------------------------
0 | dcc_criterion | MSELoss | 0      | train
1 | encoder       | MLP     | 206 K  | train
2 | decoder       | MLP     | 207 K  | train
--------------------------------------

NameError: name 'exit' is not defined

## 7. Experiment: Train UMAP Multiple Times

In [ ]:
print("=" * 80)
print(f"STEP 2: Train UMAP {n_runs} times with different random seeds")
print("=" * 80)

umap_embeddings_list = []
umap_seeds = [base_seed + i * 1000 for i in range(n_runs)]

for run_idx, seed in enumerate(tqdm(umap_seeds, desc="Training UMAP runs")):
    print(f"\nRun {run_idx + 1}/{n_runs} (seed={seed})")
    
    # Train UMAP
    umap_model = umap.UMAP(
        n_components=latent_dim,
        n_neighbors=umap_n_neighbors,
        min_dist=umap_min_dist,
        metric=umap_metric,
        random_state=seed,
        verbose=False
    )
    
    embeddings = umap_model.fit_transform(X_full)
    umap_embeddings_list.append(embeddings)
    print(f"  Embeddings shape: {embeddings.shape}")

print(f"\n✓ Trained {n_runs} UMAP models")
print(f"  Each embedding: {umap_embeddings_list[0].shape}")

STEP 2: Train UMAP 10 times with different random seeds


Training UMAP runs:   0%|          | 0/10 [00:00<?, ?it/s]


Run 1/10 (seed=20210321)


Exception ignored on calling ctypes callback function: <function ExecutionEngine._raw_object_cache_notify at 0x7f9cb3357740>
Traceback (most recent call last):
  File "/home/ogoodwin/projects/geoencoder_sasha/.venv/lib/python3.12/site-packages/llvmlite/binding/executionengine.py", line 178, in _raw_object_cache_notify
    def _raw_object_cache_notify(self, data):

KeyboardInterrupt: 


## 8. Compute Pairwise Correlations

In [ ]:
print("=" * 80)
print(f"STEP 2: Train UMAP {n_runs} times with different random seeds")
print("=" * 80)

umap_embeddings_list = []
umap_seeds = [base_seed + i * 1000 for i in range(n_runs)]

for run_idx, seed in enumerate(tqdm(umap_seeds, desc="Training UMAP runs")):
    print(f"\nRun {run_idx + 1}/{n_runs} (seed={seed})")
    
    # Train UMAP
    umap_model = umap.UMAP(
        n_components=latent_dim,
        n_neighbors=umap_n_neighbors,
        min_dist=umap_min_dist,
        metric=umap_metric,
        random_state=seed,
        verbose=False
    )
    
    embeddings = umap_model.fit_transform(X_full)
    umap_embeddings_list.append(embeddings)
    print(f"  Embeddings shape: {embeddings.shape}")

print(f"\n✓ Trained {n_runs} UMAP models")
print(f"  Each embedding: {umap_embeddings_list[0].shape}")

## 9. Compute Per-OA Standard Deviations

In [ ]:
print("=" * 80)
print("STEP 3: Compute pairwise correlations between runs")
print("=" * 80)

# AE correlations
print("\nComputing AE pairwise correlations...")
ae_mean_corr, ae_corr_matrix, ae_all_corrs = compute_pairwise_correlations(ae_embeddings_list)

# UMAP correlations
print("Computing UMAP pairwise correlations...")
umap_mean_corr, umap_corr_matrix, umap_all_corrs = compute_pairwise_correlations(umap_embeddings_list)

print("\n" + "=" * 80)
print("PAIRWISE CORRELATION RESULTS")
print("=" * 80)

print(f"\nAutoencoder (AE):")
print(f"  Mean pairwise correlation: {ae_mean_corr:.6f}")
print(f"  Std pairwise correlation: {ae_all_corrs.std():.6f}")
print(f"  Min correlation: {ae_all_corrs.min():.6f}")
print(f"  Max correlation: {ae_all_corrs.max():.6f}")

print(f"\nUMAP:")
print(f"  Mean pairwise correlation: {umap_mean_corr:.6f}")
print(f"  Std pairwise correlation: {umap_all_corrs.std():.6f}")
print(f"  Min correlation: {umap_all_corrs.min():.6f}")
print(f"  Max correlation: {umap_all_corrs.max():.6f}")

print(f"\nComparison:")
print(f"  AE is {ae_mean_corr - umap_mean_corr:.4f} more correlated than UMAP")
print(f"  AE is {umap_all_corrs.std() / ae_all_corrs.std():.2f}x more stable (lower std)")

## 10. Visualization: Comparison Table

In [ ]:
print("=" * 80)
print("STEP 4: Compute per-OA standard deviations across runs")
print("=" * 80)

# Compute per-OA std
print("\nComputing AE per-OA std...")
ae_per_oa_std = compute_per_oa_std(ae_embeddings_list)

print("Computing UMAP per-OA std...")
umap_per_oa_std = compute_per_oa_std(umap_embeddings_list)

print("\n" + "=" * 80)
print("PER-OA STANDARD DEVIATION RESULTS")
print("=" * 80)

print(f"\nAutoencoder (AE):")
print(f"  Mean per-OA std: {ae_per_oa_std.mean():.6f}")
print(f"  Median per-OA std: {np.median(ae_per_oa_std):.6f}")
print(f"  Max per-OA std: {ae_per_oa_std.max():.6f}")

print(f"\nUMAP:")
print(f"  Mean per-OA std: {umap_per_oa_std.mean():.6f}")
print(f"  Median per-OA std: {np.median(umap_per_oa_std):.6f}")
print(f"  Max per-OA std: {umap_per_oa_std.max():.6f}")

# Save results
results = {
    'ae_embeddings': ae_embeddings_list,
    'umap_embeddings': umap_embeddings_list,
    'ae_mean_corr': ae_mean_corr,
    'umap_mean_corr': umap_mean_corr,
    'ae_per_oa_std': ae_per_oa_std,
    'umap_per_oa_std': umap_per_oa_std,
    'oa_ids': oa_ids
}

with open(f"{output_dir}/data/stability_results.pkl", 'wb') as f:
    pickle.dump(results, f)

print(f"\n✓ Results saved to {output_dir}/data/stability_results.pkl")

## 11. Visualization: Pairwise Correlation Distributions

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    'Metric': [
        'Mean Pairwise Correlation',
        'Std Pairwise Correlation',
        'Min Pairwise Correlation',
        'Max Pairwise Correlation',
        'Mean Per-OA Std',
        'Median Per-OA Std',
        'Max Per-OA Std'
    ],
    'AE': [
        ae_mean_corr,
        ae_all_corrs.std(),
        ae_all_corrs.min(),
        ae_all_corrs.max(),
        ae_per_oa_std.mean(),
        np.median(ae_per_oa_std),
        ae_per_oa_std.max()
    ],
    'UMAP': [
        umap_mean_corr,
        umap_all_corrs.std(),
        umap_all_corrs.min(),
        umap_all_corrs.max(),
        umap_per_oa_std.mean(),
        np.median(umap_per_oa_std),
        umap_per_oa_std.max()
    ]
})

print("\n" + "=" * 80)
print("DETAILED COMPARISON TABLE")
print("=" * 80)
print(comparison_df.to_string(index=False))

# Save table
comparison_df.to_csv(f"{output_dir}/data/stability_comparison_table.csv", index=False)
print(f"\n✓ Table saved to {output_dir}/data/stability_comparison_table.csv")

## 12. Visualization: Per-OA Standard Deviation Histograms

In [ ]:
# Plot pairwise correlation distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(ae_all_corrs, bins=20, alpha=0.6, color='steelblue', 
             label=f'AE (mean={ae_mean_corr:.4f})', edgecolor='black')
axes[0].hist(umap_all_corrs, bins=20, alpha=0.6, color='coral', 
             label=f'UMAP (mean={umap_mean_corr:.4f})', edgecolor='black')
axes[0].axvline(ae_mean_corr, color='blue', linestyle='--', linewidth=2)
axes[0].axvline(umap_mean_corr, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Pairwise Correlation', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Pairwise Correlations\nBetween Runs', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 1])

# Box plot
data_to_plot = [ae_all_corrs, umap_all_corrs]
bp = axes[1].boxplot(data_to_plot, labels=['AE', 'UMAP'], patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('coral')
axes[1].set_ylabel('Pairwise Correlation', fontsize=12)
axes[1].set_title('Pairwise Correlation Distribution\n(Box Plot)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig(f"{output_dir}/pairwise_correlation_distributions.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Pairwise correlation plots saved")

## 13. Visualization: Geographic Map of High-Variance Areas

In [ ]:
# Plot per-OA std distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(ae_per_oa_std, bins=50, alpha=0.6, color='steelblue',
             label=f'AE (mean={ae_per_oa_std.mean():.4f})', edgecolor='black')
axes[0].hist(umap_per_oa_std, bins=50, alpha=0.6, color='coral',
             label=f'UMAP (mean={umap_per_oa_std.mean():.4f})', edgecolor='black')
axes[0].axvline(ae_per_oa_std.mean(), color='blue', linestyle='--', linewidth=2)
axes[0].axvline(umap_per_oa_std.mean(), color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Per-OA Standard Deviation', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Per-OA Std Across Runs', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Box plot
data_to_plot = [ae_per_oa_std, umap_per_oa_std]
bp = axes[1].boxplot(data_to_plot, labels=['AE', 'UMAP'], patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('coral')
axes[1].set_ylabel('Per-OA Standard Deviation', fontsize=12)
axes[1].set_title('Per-OA Std Distribution\n(Box Plot)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{output_dir}/per_oa_std_distributions.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Per-OA std plots saved")

## 14. Summary and Interpretation

In [ ]:
# Load geometry (if available)
try:
    print("Loading geometry for mapping...")
    gdf = gpd.read_file(geofile_path)
    gdf = gdf.set_index('OA21CD')  # Adjust column name if needed
    
    # Merge with std data
    std_df = pd.DataFrame({
        'OA': oa_ids,
        'AE_std': ae_per_oa_std,
        'UMAP_std': umap_per_oa_std
    }).set_index('OA')
    
    gdf_merged = gdf.join(std_df, how='inner')
    
    # Plot UMAP std map
    fig, axes = plt.subplots(1, 2, figsize=(16, 10))
    
    # AE map
    gdf_merged.plot(
        column='AE_std',
        cmap='YlOrRd',
        linewidth=0,
        ax=axes[0],
        legend=True,
        legend_kwds={'label': 'Std Deviation', 'shrink': 0.8}
    )
    axes[0].set_title(f'AE: Per-OA Std Across {n_runs} Runs\nMean={ae_per_oa_std.mean():.4f}', 
                      fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # UMAP map
    gdf_merged.plot(
        column='UMAP_std',
        cmap='YlOrRd',
        linewidth=0,
        ax=axes[1],
        legend=True,
        legend_kwds={'label': 'Std Deviation', 'shrink': 0.8}
    )
    axes[1].set_title(f'UMAP: Per-OA Std Across {n_runs} Runs\nMean={umap_per_oa_std.mean():.4f}', 
                      fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    plt.suptitle('Geographic Distribution of Embedding Variance\nHigher values = less stable', 
                 fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/variance_map.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Variance map saved")
    
except FileNotFoundError:
    print(f"⚠ Geometry file not found at {geofile_path}")
    print("  Skipping geographic map visualization")
except Exception as e:
    print(f"⚠ Error creating map: {e}")
    print("  Skipping geographic map visualization")

## 15. Summary and Interpretation

In [ ]:
print("=" * 80)
print("EXPERIMENT SUMMARY: RETRAINING STABILITY")
print("=" * 80)

print(f"\n📊 Experimental Setup:")
print(f"  - Number of runs: {n_runs}")
print(f"  - Embedding dimension: {latent_dim}D")
print(f"  - Training epochs: {n_epochs}")
print(f"  - OAs analyzed: {len(oa_ids)}")

print(f"\n📈 Key Findings:")

print(f"\n1. Mean Pairwise Correlation (between runs):")
print(f"   AE:   {ae_mean_corr:.6f}")
print(f"   UMAP: {umap_mean_corr:.6f}")
print(f"   → AE is {ae_mean_corr - umap_mean_corr:.4f} more correlated")

print(f"\n2. Mean Per-OA Standard Deviation:")
print(f"   AE:   {ae_per_oa_std.mean():.6f}")
print(f"   UMAP: {umap_per_oa_std.mean():.6f}")
print(f"   → UMAP is {umap_per_oa_std.mean() / ae_per_oa_std.mean():.2f}x more variable")

print(f"\n💡 Interpretation:")
if ae_mean_corr > 0.95:
    print(f"   ✓ AE shows EXCELLENT reproducibility (r > 0.95)")
    print(f"     Different researchers will get nearly identical results")
elif ae_mean_corr > 0.90:
    print(f"   ✓ AE shows GOOD reproducibility (r > 0.90)")
else:
    print(f"   ⚠ AE shows MODERATE reproducibility (r = {ae_mean_corr:.3f})")

if umap_mean_corr < 0.80:
    print(f"   ✗ UMAP shows POOR reproducibility (r < 0.80)")
    print(f"     Results vary significantly between runs")
elif umap_mean_corr < 0.90:
    print(f"   ⚠ UMAP shows MODERATE reproducibility (r = {umap_mean_corr:.3f})")
else:
    print(f"   ✓ UMAP shows GOOD reproducibility (r > 0.90)")

# Identify high-variance areas for UMAP
high_variance_threshold = np.percentile(umap_per_oa_std, 90)
high_variance_oas = oa_ids[umap_per_oa_std > high_variance_threshold]
print(f"\n3. High-Variance Areas (UMAP):")
print(f"   {len(high_variance_oas)} OAs ({len(high_variance_oas)/len(oa_ids)*100:.1f}%) have high embedding variance")
print(f"   These are likely rural or unusual areas with sparse manifold structure")

print(f"\n🎯 Conclusion:")
print(f"   AE is {ae_mean_corr - umap_mean_corr:.3f} more reproducible than UMAP.")
print(f"   This demonstrates AE's advantage for scientific research where")
print(f"   reproducibility is critical for peer review and replication studies.")

print(f"\n📁 All results saved to: {output_dir}")
print("=" * 80)